In [2]:
# ============================================================
# Silver Layer - Cleaning and Standardisation
# Fleet Data Platform Modernisation
# ============================================================

storage_account = "stfleetcndatalakeprod"
bronze_delta_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/delta/"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/fleet_telematics/"

print(f"Reading from: {bronze_delta_path}")
print(f"Writing to: {silver_path}")

StatementMeta(sparkpool01, 6, 2, Finished, Available, Finished, False)

Reading from: abfss://bronze@stfleetcndatalakeprod.dfs.core.windows.net/delta/
Writing to: abfss://silver@stfleetcndatalakeprod.dfs.core.windows.net/fleet_telematics/


In [3]:
from pyspark.sql import functions as F

df_bronze = spark.read.format("delta").load(bronze_delta_path)

print(f"Records from Bronze: {df_bronze.count()}")
df_bronze.printSchema()

StatementMeta(sparkpool01, 6, 3, Finished, Available, Finished, False)

Records from Bronze: 100
root
 |-- event_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- speed_mph: double (nullable = true)
 |-- fuel_level_pct: double (nullable = true)
 |-- engine_temperature_c: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [4]:
df_silver = df_bronze \
    .withColumn("speed_mph", F.col("speed_mph").cast("double")) \
    .withColumn("fuel_level_pct", F.col("fuel_level_pct").cast("double")) \
    .withColumn("engine_temperature_c", F.col("engine_temperature_c").cast("double")) \
    .withColumn("latitude", F.col("latitude").cast("double")) \
    .withColumn("longitude", F.col("longitude").cast("double")) \
    .withColumn("timestamp", F.to_timestamp(F.col("timestamp"), "yyyy-MM-dd HH:mm:ss")) \
    .dropDuplicates(["event_id"]) \
    .dropna(subset=["vehicle_id", "event_id"]) \
    .withColumn("is_overspeeding", F.col("speed_mph") > 60) \
    .withColumn("ingestion_audit_timestamp", F.current_timestamp())

print(f"Records after cleaning: {df_silver.count()}")
df_silver.printSchema()
df_silver.show(5)

StatementMeta(sparkpool01, 6, 4, Finished, Available, Finished, False)

Records after cleaning: 100
root
 |-- event_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- speed_mph: double (nullable = true)
 |-- fuel_level_pct: double (nullable = true)
 |-- engine_temperature_c: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- is_overspeeding: boolean (nullable = true)
 |-- ingestion_audit_timestamp: timestamp (nullable = false)

+---------+----------+-------------------+---------+--------------+--------------------+---------+---------+--------------------+--------------------+---------------+-------------------------+
| event_id|vehicle_id|          timestamp|speed_mph|fuel_level_pct|engine_temperature_c| latitude|longitude| ingestion_timestamp|         source_file|is_overspeeding|ingestion_audit_timestamp|
+---------+----------+------

In [5]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print(f"Silver Delta written successfully")
print(f"Total clean records: {df_silver.count()}")

StatementMeta(sparkpool01, 6, 5, Finished, Available, Finished, False)

Silver Delta written successfully
Total clean records: 100
